In [0]:
import dlt
import pyspark.sql.functions as F

catalog = "meridian_tNN" # Replace NN

# 1. Read Bronze Fact and Lookup
@dlt.view(name="bronze_returns_stream")
def bronze_returns_stream():
    return spark.readStream.table(f"{catalog}.bronze.bronze_store_returns")

@dlt.view(name="sales_lookup")
def sales_lookup():
    return spark.read.table(f"{catalog}.bronze.bronze_store_sales")

# 2. Cleansing, Casting, and Orphan Detection
@dlt.view(name="returns_cleansed")
def returns_cleansed():
    df_returns = dlt.read_stream("bronze_returns_stream")
    df_sales = dlt.read("sales_lookup")
    
    # Left join to detect orphans (returns without matching sales)
    joined = df_returns.join(
        F.broadcast(df_sales),
        (df_returns.sr_item_sk == df_sales.ss_item_sk) & 
        (df_returns.sr_ticket_number == df_sales.ss_ticket_number),
        "left"
    )
    
    return joined.select(
        df_returns.sr_item_sk.cast("int").alias("item_sk"),
        df_returns.sr_customer_sk.cast("int").alias("customer_sk"),
        df_returns.sr_ticket_number.cast("int").alias("ticket_number"),
        df_returns.sr_return_quantity.cast("int").alias("return_quantity"),
        df_returns.sr_refunded_cash.cast("decimal(10,2)").alias("refunded_cash"),
        df_returns.sr_reason_sk.cast("int").alias("reason_sk"),
        # Flag orphan returns (Advanced Challenge)
        F.when(df_sales.ss_item_sk.isNull(), True).otherwise(False).alias("is_orphan_return"),
        # Determine rule violations for quarantine
        F.when(F.col("sr_item_sk").isNull() | (F.col("sr_return_quantity") <= 0), False).otherwise(True).alias("is_valid")
    )

# 3. Valid Silver Table (Deduplicated)
@dlt.table(
    name="silver_store_returns",
    comment="Validated, deduplicated fact table"
)
@dlt.expect("valid_quantity", "return_quantity > 0") # Keep and record
@dlt.expect("valid_item", "item_sk IS NOT NULL") # Keep and record
def silver_store_returns():
    return (
        dlt.read_stream("returns_cleansed")
        .filter("is_valid = True")
        .drop("is_valid")
        # Deduplication (Step 5)
        .dropDuplicates(["ticket_number", "item_sk"]) 
    )

# 4. Quarantine Table (Ops schema)
@dlt.table(
    name="quarantine_store_returns",
    comment="Invalid records dropped from Silver",
    table_properties={"pipelines.autoOptimize.targetSchema": f"{catalog}.ops"}
)
def quarantine_store_returns():
    return (
        dlt.read_stream("returns_cleansed")
        .filter("is_valid = False")
        .drop("is_valid")
    )